In [ ]:
import yfinance as yf

from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool

In [ ]:
load_dotenv()

In [ ]:
model = ChatGroq(
    model = 'llama-3.3-70b-versatile',
    temperature= 0
)

In [ ]:
search_tool = DuckDuckGoSearchRun()

@tool
def calculator(fst_no: float, scd_no: float, operator: str) -> dict:
    """Perform arithmetic on two numbers. Operator must be: add, subtract, multiply, or divide."""  
    try:
        if operator == 'add':
            result = fst_no + scd_no
        elif operator == 'subtract':
            result = fst_no - scd_no
        elif operator == 'multiply':
            result = fst_no * scd_no
        elif operator == 'divide':
            if scd_no == 0:
                return {'Error': 'Division by zero is infinity!'}
            else:
                result = fst_no / scd_no
        else:
            return {'error': f'Unknown operator: {operator}'}
        
        return {'first no': fst_no, 'second no': scd_no, 'result': result}
    except Exception as e:
        return {'error': str(e)}

@tool
def stock_price_predictor(symbol: str) -> dict:
    """Fetch latest stock price for a given symbol (e.g. 'OGDC.KA', 'AAPL.KA', 'HBL.KA')."""
    if not symbol.endswith('.KA'): # Adding '.KA' suffix is essential to get stock price for PSX.
        symbol = symbol + '.KA'
        
    ticker = yf.Ticker(symbol)        
    data = ticker.history(period='1d') # Fetch stock price from last 1-day.

    if data.empty:                    
        return {'error': f'No data found for symbol {symbol}!'}

    latest = data.iloc[-1]
    return {
        'symbol': symbol,
        'price': round(latest['Close'], 3)
    }

In [ ]:
tool_list = [search_tool, calculator, stock_price_predictor]

llm_with_tools = model.bind_tools(tool_list)

In [ ]:
class ChatState(TypedDict):
    msgs : Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_node(state : ChatState):
    msgs = state['msgs']
    response = llm_with_tools.invoke(msgs)
    return {'msgs' : [response]}

tool_node = ToolNode(tool_list, messages_key='msgs')

In [ ]:
graph = StateGraph(ChatState)
graph.add_node('chat_node', chat_node)
graph.add_node('tools', tool_node)

graph.add_edge(START, 'chat_node')                        
graph.add_conditional_edges(                              
    'chat_node',
# In lamdba tool_conditon fetches last mesg and checks curr state needs 'tools' then return 'tools' otherwise 'end'.
    lambda state: tools_condition(state, messages_key='msgs')
)
# Loops result back to chat_node so llm can decide what's next.
graph.add_edge('tools', 'chat_node')                      

workflow = graph.compile()

In [ ]:
workflow

In [ ]:
user_query = workflow.invoke({'msgs' : [HumanMessage(content= 'Divide 35 by 5')]})
print(user_query['msgs'][-1].content)

In [ ]:
user_query = workflow.invoke({'msgs' : [HumanMessage(content= 'What is the stock price for HBL(Habib Bank Limited) in pkr and how much stocks can i buy under 10000 pkr ')]})
print(user_query['msgs'][-1].content)